In [1]:
import torch
from transformers import GPT2Tokenizer, GPT2Model

torch.set_grad_enabled(False)

model = GPT2Model.from_pretrained("gpt2")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model.eval()



config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

GPT2Model(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-11): 12 x GPT2Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): GPT2Attention(
        (c_attn): Conv1D(nf=2304, nx=768)
        (c_proj): Conv1D(nf=768, nx=768)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): GPT2MLP(
        (c_fc): Conv1D(nf=3072, nx=768)
        (c_proj): Conv1D(nf=768, nx=3072)
        (act): NewGELUActivation()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)

In [18]:
#Tokenizador

input_text = "Hello, good morning"

encoded = tokenizer(input_text, return_tensors="pt")
print(encoded)

ids_input = encoded["input_ids"]

num_tokens = ids_input.shape[1]

print("tokens: ", tokenizer.convert_ids_to_tokens(ids_input[0]))


{'input_ids': tensor([[15496,    11,   922,  3329]]), 'attention_mask': tensor([[1, 1, 1, 1]])}
tokens:  ['Hello', ',', 'Ġgood', 'Ġmorning']
tensor(11)


In [37]:
#Cálculo de wpe y wte

wpe = model.wpe.weight
wte = model.wte.weight

positions = torch.arange(num_tokens)
embeddings = []

for i in range(num_tokens):
    embeddings.append(wte[ids_input[0][i]] + wpe[positions[i]])

embeddings = torch.stack(embeddings)

print(embeddings)


tensor([[-0.0875, -0.3301,  0.0152,  ...,  0.0285, -0.0014,  0.0068],
        [ 0.0355, -0.0567, -0.0626,  ...,  0.0619, -0.0195, -0.0601],
        [-0.0796, -0.1353,  0.0913,  ..., -0.1339, -0.0702, -0.1327],
        [ 0.0633,  0.0460,  0.1728,  ...,  0.0879,  0.0270, -0.1788]])


In [38]:
#Huggingface embeddings results

salida_modelo = model(ids_input, output_hidden_states=True)

embeddings_modelo = salida_modelo.hidden_states[0]

print(embeddings)

tensor([[-0.0875, -0.3301,  0.0152,  ...,  0.0285, -0.0014,  0.0068],
        [ 0.0355, -0.0567, -0.0626,  ...,  0.0619, -0.0195, -0.0601],
        [-0.0796, -0.1353,  0.0913,  ..., -0.1339, -0.0702, -0.1327],
        [ 0.0633,  0.0460,  0.1728,  ...,  0.0879,  0.0270, -0.1788]])


In [41]:
def comparar(tensor_1, tensor_2):
    diferencias = (tensor_1 - tensor_2).abs()
    diferencia_max = diferencias.max().item()

    print(f"Diferencia máxima: {diferencia_max}")


comparar(embeddings, embeddings_modelo)

Diferencia máxima: 0.0
